# Evaluating the best-performing model so far on various data.

In [1]:
from transformers import pipeline


2025-06-09 12:41:32.738035: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749472892.926015      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749472892.979047      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
# Model
pipe_large_v3 = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3", return_timestamps=True)


## twos.wav

In [ ]:
result_twos = pipe_large_v3("twos.wav")
result_twos


/usr/local/lib/python3.11/dist-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.


{'text': " Hey Jamie, are you hungry? I'm quite hungry. What should we cook? Oh my god Alex, I have this recipe idea. Are you down? Yeah, tell me. This chicken biryani, that is spicy. I don't know if you can tell the spiciness. Yeah, I'll be fine. I think I can. Go on. So you marinate the chicken, add enough chili powder and let it sit for 20 minutes. Oh wow, that sounds yummy. Yes. After you pan fry it and mix it with rice. Okay, okay. I'll try that. Let's do that then. Yeah, please let me know how it goes. I will. Thank you.",
 'chunks': [{'timestamp': (0.0, 5.0),
   'text': " Hey Jamie, are you hungry? I'm quite hungry. What should we cook?"},
  {'timestamp': (5.0, 10.0),
   'text': ' Oh my god Alex, I have this recipe idea. Are you down?'},
  {'timestamp': (10.0, 11.0), 'text': ' Yeah, tell me.'},
  {'timestamp': (11.0, 19.0),
   'text': " This chicken biryani, that is spicy. I don't know if you can tell the spiciness."},
  {'timestamp': (19.0, 22.0),
   'text': " Yeah, I'll be fin

In [2]:
!pip install evaluate 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.8.4.1 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cudnn-cu12==9.1.0.70; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn-cu12 9.3.0.75 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cufft-cu12==1

In [1]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.2 MB/s eta 0:00:00a 0:00:01


In [3]:
import evaluate

2025-04-28 10:21:52.070910: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745835712.294544      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745835712.353212      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [8]:
# Load metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

# Ground truth and predicted transcription
reference = """
Hey Jamie, are you hungry? I'm quite hungry. what should we cook? Oh my god! Alex, I have this recipe idea. Are you down? Yeah, tell me. There is this chicken biryani. Mhmm. That is spicy, I don’t know if you can tolerate the spiciness. Yeah, I’ll be fine, I think I can. Go on. So you marinate the chicken, add enough chilli powder, and let it sit for 20 minutes. Oh wow, that sounds yummy. Yes. After you pan fry it and mix it with rice. Okay, okay, I'll try that. Let’s do that then. Yeah, please let me know how it goes. I will. thank you.
"""
prediction = """
Hey Jamie, are you hungry? I'm quite hungry. What should we cook? Oh my god Alex, I have this recipe idea. Are you down? Yeah, tell me. This chicken biryani, that is spicy. I don't know if you can tell the spiciness. Yeah, I'll be fine. I think I can. Go on. So you marinate the chicken, add enough chili powder and let it sit for 20 minutes. Oh wow, that sounds yummy. Yes. After you pan fry it and mix it with rice. Okay, okay. I'll try that. Let's do that then. Yeah, please let me know how it goes. I will. Thank you.
"""

# Clean and normalize text
reference = " ".join(reference.lower().split())
prediction = " ".join(prediction.lower().split())

# Compute scores
wer = wer_metric.compute(predictions=[prediction], references=[reference])
cer = cer_metric.compute(predictions=[prediction], references=[reference])

print(f"WER: {wer:.2%}")
print(f"CER: {cer:.2%}")

WER: 12.96%
CER: 5.71%


In [2]:
# With timestamps
pipe_large_v3 = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3", return_timestamps="word")


config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Device set to use cuda:0


In [4]:
result_word_timstamps = pipe_large_v3(r"/kaggle/input/two-wav/twos.wav")
result_word_timstamps


/usr/local/lib/python3.11/dist-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


{'text': " Hey Jamie, are you hungry? I'm quite hungry. What should we cook? Oh my god Alex, I have this recipe idea. Are you down? Yeah, tell me. This chicken biryani, that is spicy. I don't know if you can tell the spiciness. Yeah, I'll be fine. I think I can. Go on. So you marinate the chicken, add enough chili powder and let it sit for 20 minutes. Oh wow, that sounds yummy. Yes. After you pan fry it and mix it with rice. Okay, okay. I'll try that. Let's do that then. Yeah, please let me know how it goes. I will. Thank you.",
 'chunks': [{'text': ' Hey', 'timestamp': (0.0, 0.64)},
  {'text': ' Jamie,', 'timestamp': (0.64, 1.2)},
  {'text': ' are', 'timestamp': (1.2, 1.72)},
  {'text': ' you', 'timestamp': (1.72, 1.82)},
  {'text': ' hungry?', 'timestamp': (1.82, 2.24)},
  {'text': " I'm", 'timestamp': (2.24, 2.44)},
  {'text': ' quite', 'timestamp': (2.44, 2.8)},
  {'text': ' hungry.', 'timestamp': (2.8, 3.34)},
  {'text': ' What', 'timestamp': (3.34, 3.9)},
  {'text': ' should', 't

In [14]:
import json

In [15]:
# Convert timestamps from tuples to lists (for JSON compatibility)
for chunk in result_word_timstamps['chunks']:
    chunk['timestamp'] = list(chunk['timestamp'])

# Save to JSON file
with open('twos_words_timestamps.json', 'w', encoding='utf-8') as f:
    json.dump(result_word_timstamps, f, indent=2, ensure_ascii=False)

## Data_DeepFilterNet3.wav

In [ ]:
resul_data_deepfilternet3 = pipe_large_v3("Data_DeepFilterNet3.wav")
resul_data_deepfilternet3


/usr/local/lib/python3.11/dist-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


{'text': " Hello? I'm here. Cough, cough, cough. Why? Nothing. I'm in the kitchen. Can you tell me what? Yes. Are you talking to Molly at work? No. What are you doing then? What's the point? Welcome to Parkway!",
 'chunks': [{'timestamp': (0.0, 7.0), 'text': " Hello? I'm here."},
  {'timestamp': (7.0, 0.0), 'text': ''},
  {'timestamp': (3.0, 9.0), 'text': ' Cough, cough, cough.'},
  {'timestamp': (10.0, 11.0), 'text': ' Why? Nothing.'},
  {'timestamp': (12.0, 14.0), 'text': " I'm in the kitchen."},
  {'timestamp': (20.0, 21.0), 'text': ' Can you tell me what?'},
  {'timestamp': (21.0, 22.0), 'text': ' Yes.'},
  {'timestamp': (22.0, 24.0), 'text': ' Are you talking to Molly at work?'},
  {'timestamp': (24.0, 25.0), 'text': ' No.'},
  {'timestamp': (25.0, 27.0), 'text': ' What are you doing then?'},
  {'timestamp': (28.0, 29.0), 'text': " What's the point?"},
  {'timestamp': (0.0, 2.0), 'text': ' Welcome to Parkway!'}]}

In [13]:
# Load metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

# Ground truth and predicted transcription
reference = """
mum
hello I'm here
okay
hello
hi
where's the cigarettes
in the kitchen
the camera's on
yes
are you talking to it while you work
no 
what y' doing then
what's the point oh god look what I'm wearing
"""
prediction = """
Hello? I'm here. Cough, cough, cough. Why? Nothing. I'm in the kitchen. Can you tell me what? Yes. Are you talking to Molly at work? No. What are you doing then? What's the point? Welcome to Parkway!
"""

# Clean and normalize text
reference = " ".join(reference.lower().split())
prediction = " ".join(prediction.lower().split())

# Compute scores
wer = wer_metric.compute(predictions=[prediction], references=[reference])
cer = cer_metric.compute(predictions=[prediction], references=[reference])

print(f"WER: {wer:.2%}")
print(f"CER: {cer:.2%}")

WER: 79.49%
CER: 51.28%


In [3]:
result_word_timstamps = pipe_large_v3(r"/kaggle/input/main-audio/Data_DeepFilterNet3.wav")
result_word_timstamps

/usr/local/lib/python3.11/dist-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
WhisperModel is using WhisperSdpaAttention, but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True` or `layer_head_mask` not None. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the mod

{'text': " Hello? I'm here. Cough, cough, cough. Why? Nothing. I'm in the kitchen. Can you tell me what? Yes. Are you talking to Molly at work? No. What are you doing then? What's the point? Welcome to Parkway!",
 'chunks': [{'text': ' Hello?', 'timestamp': (29.98, 29.98)},
  {'text': " I'm", 'timestamp': (29.98, 29.98)},
  {'text': ' here.', 'timestamp': (29.98, 29.98)},
  {'text': ' Cough,', 'timestamp': (26.88, 26.88)},
  {'text': ' cough,', 'timestamp': (26.88, 26.88)},
  {'text': ' cough.', 'timestamp': (26.88, 26.88)},
  {'text': ' Why?', 'timestamp': (26.88, 26.88)},
  {'text': ' Nothing.', 'timestamp': (26.88, 26.88)},
  {'text': " I'm", 'timestamp': (26.88, 26.88)},
  {'text': ' in', 'timestamp': (26.88, 26.88)},
  {'text': ' the', 'timestamp': (26.88, 26.88)},
  {'text': ' kitchen.', 'timestamp': (26.88, 26.88)},
  {'text': ' Can', 'timestamp': (26.88, 27.4)},
  {'text': ' you', 'timestamp': (27.4, 27.62)},
  {'text': ' tell', 'timestamp': (27.62, 27.68)},
  {'text': ' me', '

##  Data.wav

In [ ]:
result_data = pipe_large_v3("Data.wav")
result_data

/usr/local/lib/python3.11/dist-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


{'text': " Hello. Hello. I'm here. Hi. Hello. Hi. What is all that? In the kitchen. Are you talking to him while you work? No. What are you doing then? What's the point? I've gone the fine way.",
 'chunks': [{'timestamp': (0.0, 2.0), 'text': ' Hello.'},
  {'timestamp': (2.0, 4.0), 'text': ' Hello.'},
  {'timestamp': (4.0, 6.0), 'text': " I'm here."},
  {'timestamp': (6.0, 8.0), 'text': ' Hi.'},
  {'timestamp': (14.0, 16.0), 'text': ' Hello.'},
  {'timestamp': (16.0, 18.0), 'text': ' Hi.'},
  {'timestamp': (18.0, 20.0), 'text': ' What is all that?'},
  {'timestamp': (20.0, 22.0), 'text': ' In the kitchen.'},
  {'timestamp': (28.0, 0.0), 'text': ''},
  {'timestamp': (7.0, 8.0),
   'text': ' Are you talking to him while you work? No.'},
  {'timestamp': (8.0, 11.0), 'text': ' What are you doing then?'},
  {'timestamp': (11.0, 12.0), 'text': " What's the point?"},
  {'timestamp': (12.0, 13.0), 'text': " I've gone the fine way."}]}

In [14]:
# Load metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

# Ground truth and predicted transcription
reference = """
mum
hello I'm here
okay
hello
hi
where's the cigarettes
in the kitchen
the camera's on
yes
are you talking to it while you work
no 
what y' doing then
what's the point oh god look what I'm wearing
"""
prediction = """
Hello. Hello. I'm here. Hi. Hello. Hi. What is all that? In the kitchen. Are you talking to him while you work? No. What are you doing then? What's the point? I've gone the fine way.
"""

# Clean and normalize text
reference = " ".join(reference.lower().split())
prediction = " ".join(prediction.lower().split())

# Compute scores
wer = wer_metric.compute(predictions=[prediction], references=[reference])
cer = cer_metric.compute(predictions=[prediction], references=[reference])

print(f"WER: {wer:.2%}")
print(f"CER: {cer:.2%}")

WER: 71.79%
CER: 41.54%


In [4]:
result_word_timstamps = pipe_large_v3(r"/kaggle/input/main-audio/Data.wav")
result_word_timstamps

/usr/local/lib/python3.11/dist-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


{'text': " Hello. Hello. I'm here. Hi. Hello. Hi. What is all that? In the kitchen. Are you talking to him while you work? No. What are you doing then? What's the point? I've gone the fine way.",
 'chunks': [{'text': ' Hello.', 'timestamp': (0.0, 1.46)},
  {'text': ' Hello.', 'timestamp': (1.64, 2.3)},
  {'text': " I'm", 'timestamp': (2.88, 4.94)},
  {'text': ' here.', 'timestamp': (4.94, 5.3)},
  {'text': ' Hi.', 'timestamp': (5.86, 6.34)},
  {'text': ' Hello.', 'timestamp': (10.1, 15.28)},
  {'text': ' Hi.', 'timestamp': (15.96, 17.02)},
  {'text': ' What', 'timestamp': (17.2, 17.24)},
  {'text': ' is', 'timestamp': (17.24, 17.44)},
  {'text': ' all', 'timestamp': (17.44, 17.66)},
  {'text': ' that?', 'timestamp': (17.66, 18.28)},
  {'text': ' In', 'timestamp': (19.38, 19.6)},
  {'text': ' the', 'timestamp': (19.6, 19.74)},
  {'text': ' kitchen.', 'timestamp': (19.74, 20.4)},
  {'text': ' Are', 'timestamp': (51.98, 51.98)},
  {'text': ' you', 'timestamp': (51.98, 51.98)},
  {'text': 

# Model Hyperparameter tuning (not complete)

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 6.12 MiB is free. Process 6706 has 14.73 GiB memory in use. Of the allocated memory 14.10 GiB is allocated by PyTorch, and 523.06 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
generate_kwargs = {
    "max_new_tokens": 512,  # Increased for typical audio lengths
    "num_beams": 3,  # Balanced between quality and speed
    "condition_on_prev_tokens": False,
    "compression_ratio_threshold": 1.5,  # Slightly more lenient
    "temperature": 0.2,  # Single value for consistency
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.7,  # More conservative on silence detection
    "return_timestamps": True,
}

In [ ]:
generate_kwargs = {
    "max_new_tokens": 448,
    "num_beams": 3,
    "condition_on_prev_tokens": False,
    "compression_ratio_threshold": 1.35,  # zlib compression ratio threshold (in token space)
    "temperature": (0.0),
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.6,
    "language": "english",
    "return_timestamps": True,
}

result = pipe(sample, generate_kwargs=generate_kwargs)


In [ ]:
result_twos = pipe("twos.wav", generate_kwargs=generate_kwargs)
result_twos


In [ ]:
resul_data_deepfilternet3 = pipe_large_v3_en("Data_DeepFilterNet3.wav")
resul_data_deepfilternet3


In [ ]:
result_data = pipe_large_v3_en("Data.wav")
result_data


# Calculating the speaker's stop time

In [8]:
def annotate_pauses(data, threshold=0.1):
    """
    Annotates pauses longer than the threshold (in seconds) in the text.
    
    Args:
        data (dict): Dictionary with 'text' and 'chunks' (with timestamps).
        threshold (float): Minimum pause duration to annotate (default 0.1 seconds).
    
    Returns:
        str: Annotated text with pauses inserted.
    """
    annotated_text = ""
    chunks = data['chunks']
    
    for i in range(len(chunks) - 1):
        current_chunk = chunks[i]
        next_chunk = chunks[i + 1]
        
        annotated_text += current_chunk['text']
        
        current_end = current_chunk['timestamp'][1]
        next_start = next_chunk['timestamp'][0]
        pause_duration = next_start - current_end

        if pause_duration > threshold:
            annotated_text += f" ({pause_duration:.1f})"
    
    # Add the last word
    annotated_text += chunks[-1]['text']
    
    return annotated_text.strip()

In [10]:
annotate_pauses(result_word_timstamps, threshold=0.1)

"Hey Jamie, are you hungry? I'm quite hungry. What should we cook? (1.3) Oh my god Alex, I have this recipe idea. Are you down? (0.2) Yeah, tell me. (0.4) This chicken biryani, that is spicy. I don't know if you can tell the spiciness. (0.4) Yeah, I'll be fine. I think I can. Go on. (0.4) So you marinate the chicken, add enough chili powder and let it sit for 20 minutes. (0.5) Oh wow, that sounds yummy. (0.5) Yes. (0.9) After you pan fry it and mix it with rice. (0.9) Okay, okay. (0.5) I'll try that. Let's do that then. (0.9) Yeah, please let me know how it goes. (0.2) I will. (0.4) Thank you."

In [5]:
def annotate_pauses(data, threshold=0.1):
    """
    Annotates pauses in the text:
    - Pauses between 0.08 and 0.2 seconds are annotated as (.)
    - Pauses longer than 0.2 seconds are annotated with actual duration (e.g., (0.5))

    Args:
        data (dict): Dictionary with 'text' and 'chunks' (with timestamps).
        threshold (float): Minimum pause duration to annotate (default 0.1 seconds).
    
    Returns:
        str: Annotated text with pauses inserted.
    """
    
    annotated_text = ""
    chunks = data['chunks']

    for i in range(len(chunks) - 1):
        current_chunk = chunks[i]
        next_chunk = chunks[i + 1]
        
        annotated_text += current_chunk['text']
        
        current_end = current_chunk['timestamp'][1]
        next_start = next_chunk['timestamp'][0]
        pause_duration = next_start - current_end

        if 0.08 <= pause_duration <= 0.2:
            annotated_text += " (.)"
        elif pause_duration > 0.2:
            annotated_text += f" ({pause_duration:.1f})"

    # Add the last word
    annotated_text += chunks[-1]['text']
    
    return annotated_text.strip()

In [6]:
annotate_pauses(result_word_timstamps, threshold=0.1)

"Hey Jamie, are you hungry? I'm quite hungry. What should we cook? (1.3) Oh my god Alex, I have this recipe idea. Are you down? (0.2) Yeah, tell me. (0.4) This chicken biryani, that is spicy. I don't know if you can tell the spiciness. (0.4) Yeah, I'll be fine. I think I can. Go on. (0.4) So you marinate the chicken, add enough chili powder and let it sit for 20 minutes. (0.5) Oh wow, that sounds yummy. (0.5) Yes. (0.9) After you pan fry it and mix it with rice. (0.9) Okay, okay. (0.5) I'll try that. (.) Let's do that then. (0.9) Yeah, please let me know how it goes. (.) I will. (0.4) Thank you."

# Calculating the Loudness annotation

In [ ]:
!pip install opensmile

In [8]:
import opensmile
# Step 1: Extract loudness with openSMILE
#audio_path = "twos.wav"
audio_path = r"/kaggle/input/two-wav/twos.wav"
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.LowLevelDescriptors,
)

features = smile.process_file(audio_path)

if "Loudness_sma3" not in features.columns:
    raise ValueError("pcm_loudness not found. Check your openSMILE feature set.")

loudness = features["Loudness_sma3"]

# Step 2: Compute overall average loudness

smile_func = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

avg_loudness_audio = smile_func.process_file(audio_path)['loudness_sma3_amean'].iloc[0]

# Step 3: Compute per-word loudness & annotate if emphasized
annotated_words = []

for chunk in result_word_timstamps['chunks']:
    start, end = chunk["timestamp"][0], chunk["timestamp"][1]

    # Get word loudness base on word timestamp
    condition_get_loudness = (loudness.index.get_level_values(1).total_seconds() >= start) & (loudness.index.get_level_values(1).total_seconds() <= end)
    segment =  loudness[condition_get_loudness]
    word_loudness = segment.mean() if not segment.empty else 0

    is_emphasized = word_loudness > avg_loudness_audio

    annotated_words.append({
        "word": chunk["text"],
        "emphasized": is_emphasized,
    })

# Step 4: Print output
annotated_text = "".join(
    f"{item['word'].upper() if item['emphasized'] else item['word'].lower()}"
    for item in annotated_words
)

print(annotated_text)

 hey JAMIE, are YOU HUNGRY? I'M quite HUNGRY. what SHOULD WE cook? oh MY GOD ALEX, i HAVE THIS recipe IDEA. are YOU DOWN? YEAH, TELL ME. THIS chicken BIRYANI, that IS SPICY. i DON'T KNOW IF you CAN TELL THE spiciness. YEAH, I'LL BE FINE. I THINK I CAN. go ON. so YOU marinate THE chicken, add enough CHILI POWDER and let IT sit for 20 minutes. oh WOW, THAT SOUNDS YUMMY. yes. after YOU PAN fry IT and mix it with RICE. OKAY, okay. I'LL TRY THAT. LET'S DO THAT THEN. yeah, please let ME know HOW IT goes. I WILL. thank YOU.
